In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df=pd.read_csv("/content/Walmart.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/Walmart.csv'

In [ ]:
df.dtypes

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
print("Data Shape", df.shape)

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")
df['Quantity'] = df['Quantity'].astype(int)
df["Order Date"]

In [ ]:
df = df.drop_duplicates()
df

# Check missing values

In [ ]:
print("\nMissing Values Before Cleaning:\n", df.isnull().sum())

## Fill missing numeric values with median and categorical with mode

In [ ]:
for col in df.columns:
    if df[col].dtype in ["int64", "float64"]:
        df[col].fillna(df[col].median(), inplace=True)
    else:
        df[col].fillna(df[col].mode()[0], inplace=True)

In [ ]:
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Weekday"] = df["Order Date"].dt.day_name()
df["Profit Margin"] = (df["Profit"] / df["Sales"]) * 100

print(df[['Order Date', 'Year', 'Month', 'Weekday', 'Profit', 'Sales', 'Profit Margin']].head())


In [ ]:
# Keep a copy of the original DataFrame before one-hot encoding for plotting
df_original = df.copy()

df = pd.get_dummies(df, drop_first=True)
df.dtypes

In [ ]:
print(df.isnull().sum())


# Data Visualization

## Line Plot (Sales trend over time)

In [ ]:
plt.figure(figsize=(12,6))
df.groupby("Order Date")["Sales"].sum().plot(kind='line')
plt.title("Total Weekly Sales Over Time")
plt.xlabel("Order Date")
plt.ylabel("Weekly Sales")
plt.grid(True)
plt.show()

## Bar Plot (Average sales per category)

In [ ]:
plt.figure(figsize=(10,6))
# Use the original DataFrame before one-hot encoding for plotting
store_sales = df_original.groupby("Category")["Sales"].mean().sort_values()
store_sales.plot(kind="bar")
plt.title("Average Sales per Category")
plt.xlabel("Category")
plt.ylabel("Average Sales")
plt.show()

## Histogram (Distribution of sales)

In [ ]:
plt.figure(figsize=(10,6))
plt.hist(df["Sales"], bins=30, edgecolor='black')
plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.show()

## Boxplot (Outlier detection of sales by store)

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(x="Category", y="Sales", data=df_original)
plt.title("Outlier Detection: Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.show()

## Sales by Weekday (Bar plot)

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    x="Month",
    y="Sales",
    data=df,
    estimator="sum"
)
plt.title("Total Sales by Month")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.show()

# Univariate Analysis

In [ ]:

for col in df.select_dtypes(include=["int64", "float64"]).columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

## Part 2

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Load dataset
df = pd.read_csv("/content/Walmart.csv")

# Select only numerical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
print("Numeric Columns:", num_cols)

# --- Normalization ---
scaler_minmax = MinMaxScaler()
df_normalized = df.copy()
df_normalized[num_cols] = scaler_minmax.fit_transform(df[num_cols])

print("\nAfter Normalization (first 5 rows):")
print(df_normalized[num_cols].head())

# --- Standardization ---
scaler_standard = StandardScaler()
df_standardized = df.copy()
df_standardized[num_cols] = scaler_standard.fit_transform(df[num_cols])

print("\nAfter Standardization (first 5 rows):")
print(df_standardized[num_cols].head())


In [ ]:
df.columns


In [ ]:
df["Order ID"] = df["Order ID"].str.replace('CA-', '', regex=False).str.replace('US', '', regex=False).str.replace('-', '', regex=False)


In [ ]:
print(df.columns.tolist())

In [ ]:
df.columns

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Make a copy
df_encoded = df.copy()

# Find categorical columns
cat_cols = df_encoded.select_dtypes(include=['object']).columns
print("Categorical Columns:", cat_cols)

# Apply Label Encoding
le = LabelEncoder()
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])

print("\nAfter Label Encoding:")
print(df_encoded.head())


In [ ]:
df.columns

In [ ]:
print(df.columns.tolist())


In [ ]:
import pandas as pd

# Convert 'Date' column to datetime
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors='coerce')

# Extract features from date
df["Year1"] = df["Ship Date"].dt.year
df["Month1"] = df["Ship Date"].dt.month
df["Day1"] = df["Ship Date"].dt.day
df["Weekday1"] = df["Ship Date"].dt.weekday  # Monday=0, Sunday=6

# Drop the original date column (string not needed anymore)

df = df.drop(columns=["Order Date"])
df = df.drop(columns=["Ship Date"])


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# ---------- Step 1: Define X and y ----------
X = df.drop('Sales', axis=1)   # Independent variables
y = df['Sales']                # Dependent variable (target)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

# ---------- Step 2: Train-Test Split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=40
)

print("\nShapes after Train-Test Split:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# Separate categorical and numeric columns
cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns

# Preprocessor: encode categorical, keep numeric as is
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="passthrough"
)

# Full pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("rf", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
print("Predictions:", y_pred)

#

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


# ---------- Step 1: Define X and y ----------
X = df.drop('Sales', axis=1)   # Independent variables
y = df['Sales']                # Dependent variable (target)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

# ---------- Step 2: Train-Test Split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=40
)

print("\nShapes after Train-Test Split:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np # Import numpy for square root

# Separate categorical and numerical columns
cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns

# Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="passthrough"
)

# Pipeline with RandomForest
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("rf", RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluation
print("R² Score:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred))) # Calculate square root for RMSE

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Weekly Sales")
plt.show()
